# REAL 정책 inference — LeRobot 체크포인트로 SG2 자율 구동

VR 텔레오퍼레이션이 아니라, 학습된 LeRobot 정책(ACT부터 시작하지만 정책 종류 무관)이 카메라·관절 상태를 보고 다음 관절 목표를 스스로 예측해서 로봇을 움직입니다. Quest 컨트롤러는 필요 없습니다.

내부적으로 `PolicyDeployment`가 텔레옵과 **완전히 동일한** MuJoCo 씬/IK/충돌검사(`SG2CommandGuard`)와 REAL 세션(arming/lease/속도 pacing)을 재사용하고, `action="policy"` 경로로 정책의 예측값을 넣습니다. 충돌 검사는 텔레옵과 달리 항상 켜져 있습니다(`teleop_collision_check` 설정과 무관).

**체크포인트가 아직 없다면**: 이 노트북은 그대로 두고, 학습이 끝난 뒤 아래 "설정" 셀의 `CHECKPOINT_PATH`만 채우면 됩니다.

In [ ]:
# Locate this standalone workspace from a notebook or repository directory.
from pathlib import Path
import runpy

_v2_candidates = [candidate for base in (Path.cwd(), *Path.cwd().parents)
                  for candidate in (base, base / "VR_teleoperation")]
_v2_root = next((candidate for candidate in _v2_candidates
                 if (candidate / ".project_v2").is_file()), None)
if _v2_root is None:
    raise RuntimeError("Open this notebook from the VR_teleoperation workspace.")
_v2_bootstrap = runpy.run_path(str(_v2_root / "package" / "init_project.py"))
for _v2_name in _v2_bootstrap["__all__"]:
    globals()[_v2_name] = _v2_bootstrap[_v2_name]


## 사전 점검 (로봇 연결 없이)

먼저 터미널에서 체크포인트가 제대로 로드되는지 확인하세요. 합성 입력으로 한 번 예측해보는 스모크테스트라 로봇도 카메라도 필요 없습니다.

```bash
cd VR_teleoperation
python scripts/check_policy_inference.py <checkpoint_path>
```

`PASS`가 뜨고 관절별 예측값이 finite하게 나오면 아래로 진행하세요. 여기서 에러가 나면 로봇을 켜기 전에 먼저 잡으세요 (체크포인트 경로, lerobot 버전, device 등).

## 설정

`CHECKPOINT_PATH`만 채우면 됩니다. `HZ`는 데모 데이터를 녹화할 때(episode_recorder.py) 쓴 fps와 맞추세요 — 다르면 정책이 학습 때와 다른 속도로 관측을 받게 됩니다. `TASK`는 학습 데이터셋을 녹화할 때 쓴 task 문구와 맞추는 게 원칙입니다 (LeRobot 프레임마다 붙는 설명).

**`DRY_RUN=True`로 시작하세요.** 로봇에 실제 명령이 나가지 않고, arming까지의 흐름과 정책 예측값만 확인할 수 있습니다. 확인 후에만 `False`로 바꾸세요.

In [ ]:
from package.policy_deploy import PolicyDeployment
from package.episode_recorder import DEFAULT_CAMERAS

CHECKPOINT_PATH = ""  # 예: "/home/.../outputs/train/act_pick_and_place/checkpoints/last/pretrained_model"
ROBOT_IP = "192.168.6.2"
CAMERAS = DEFAULT_CAMERAS  # {"head": 5570, "wrist_left": 5571, "wrist_right": 5572}
HZ = 30                    # episode_recorder.py로 녹화한 fps와 맞출 것
TASK = "SG2 VR teleoperation"  # 학습 데이터셋 녹화 때 쓴 task 문구와 맞출 것
DEVICE = None              # None이면 체크포인트에 저장된 config.device 사용 (예: "cpu", "cuda", "mps"로 덮어쓰기 가능)
DRY_RUN = True             # 반드시 True로 시작. 확인 후에만 False.
DURATION_S = 60            # 한 번 실행에 최대 몇 초 (None이면 Ctrl-C 할 때까지)
COLLISION_MARGIN_M = 0.0005  # self-collision 여유값. 이미 모은 데이터(양손 근접 등) 안 막히게 넉넉히 낮춰둠 -- 실제로 부딪히는 사고가 나면 늘리세요.

assert CHECKPOINT_PATH, "CHECKPOINT_PATH를 채우세요."


## 로봇 세션 준비

[Worker 실행 안내](../docs/WORKER_SETUP.md)에 따라 bringup/outbound/inbound가 이미 실행 중이어야 합니다. 다른 Quest/텔레옵 노트북은 먼저 STOP하세요 — 정책 세션도 텔레옵과 똑같이 로봇 명령 리스(lease)를 하나만 쥘 수 있습니다.

이 셀은 체크포인트를 로드하고 로봇 상태(5560)·카메라(5570~5572) ZMQ 소켓을 엽니다. 아직 로봇에 명령을 보내지 않습니다.

In [ ]:
deploy = PolicyDeployment(
    checkpoint_path=CHECKPOINT_PATH,
    robot_ip=ROBOT_IP,
    cameras=CAMERAS,
    hz=HZ,
    task=TASK,
    device=DEVICE,
    dry_run=DRY_RUN,
    collision_margin_m=COLLISION_MARGIN_M,
)
print("policy type:", deploy.policy.policy_type, "| device:", deploy.policy.device)
print("dry_run:", deploy.dry_run, "| hz:", deploy.hz)


## 실행

`run()`은 blocking 루프입니다 — `DURATION_S`초 동안(또는 `None`이면 Ctrl-C 할 때까지) 매 틱마다 로봇 상태·카메라를 읽고, 정책 예측 → `clip_to_joint_limits`로 안전범위 clip → 손목 위치 세이프티박스(`package/safety_box.py`) FK 체크 → `action="policy"`로 전송합니다.

- 로봇 피드백이나 카메라 중 하나라도 아직 안 들어왔으면 움직이지 않고 대기합니다 (`WAIT_ROBOT` 등).
- 처음 움직이기 시작할 때 arming(세션 생성 → 로봇 브릿지 ACK 대기)이 자동으로 일어납니다.
- 예측된 관절 목표로 FK를 돌렸을 때 손목이 세이프티박스(`package/safety_box.py`)를 벗어나면, 그 스텝은 로봇에 보내지 않고 현재 위치를 유지합니다 (sim에서 `scripts/check_safety_box_inference.py`로 같은 로직을 직접 확인할 수 있습니다).
- self-collision 검사는 항상 켜져 있지만 margin을 낮춰뒀습니다(위 `COLLISION_MARGIN_M`) -- 별도 collision-IK 작업 전까지 임시 조치이니, 실기 중 이상하면 값을 올리거나 더 보강하세요.
- 충돌·관절범위·추종오차 등 어떤 이유로든 `FAULT`가 나면 루프가 자동으로 멈추고 세션을 반납합니다.
- `Ctrl-C`로 언제든 안전하게 중단할 수 있습니다 (`finally`에서 세션을 반납합니다).

In [ ]:
result = deploy.run(duration_s=DURATION_S)
print("last state:", None if result is None else (result["state"], result["reason"]))


## (선택) 키보드로 에피소드 단위 녹화+inference

VR 컨트롤러 없이 키보드만으로 에피소드 단위 롤아웃을 기록하고 싶다면 위 `deploy.run(...)` 셀 대신 이 셀을 쓰세요. 위 셀과 달리 **정책은 x를 누르기 전까지 로봇을 구동하지 않고 대기(hold)** 합니다 — x를 눌러야 그 순간부터 녹화와 policy 구동이 함께 시작됩니다.

- `x` 이 에피소드 시작: 녹화 시작 + policy 상태 reset + policy 구동 동시 시작. 다시 누르면: 녹화 저장 + policy 정지 동시 종료 (다음 x까지 로봇은 정지 상태)
- `y` 녹화 중이면 진행 중인 에피소드 버림(동시에 policy도 정지), 아니면 마지막 저장분 폐기 표시 토글
- `a` Initial pose로 안전 복귀 (guard-checked)
- `q` 또는 `Esc` 세션 종료

실행하면 별도 작은 창(`SG2 Policy Recording`)이 뜨고, 거기에 몇 번째 에피소드를 녹화 중인지·지금까지 몇 개 저장됐는지·policy가 도는 중/대기 중인지(`VREpisodeRecorder.status()` + policy 상태)와 head/wrist_left/wrist_right 카메라 3개 실시간 미리보기가 계속 갱신됩니다. 창을 못 띄우는 환경이면 자동으로 셀 출력에 상태 텍스트만 출력합니다(카메라 미리보기는 그 경우 생략).

**주의**: 학습 데이터가 아니라 정책 롤아웃 기록이므로, LeRobot 컨벤션대로 데이터셋 이름 앞에 `eval_`을 붙여 텔레옵 데이터 수집 파이프라인과 분리했습니다. 이 데이터를 학습에 그대로 섞어 넣지 마세요.

In [ ]:
from package.episode_recorder import VREpisodeRecorder, list_episodes
from package.policy_record_control import run_policy_with_keyboard_recording

EVAL_SAVE_ROOT = Path.home() / "aiworker_data"
EVAL_TASK_NAME = "eval_" + TASK.replace(" ", "_")  # eval_ prefix: policy rollout, not training data
recorder = VREpisodeRecorder(
    robot_ip         = ROBOT_IP,
    repo_id          = "local/%s" % EVAL_TASK_NAME,
    root             = EVAL_SAVE_ROOT / EVAL_TASK_NAME,
    hz               = HZ,
    max_sec          = 120,
    cameras          = CAMERAS,
    command_endpoint = "tcp://127.0.0.1:%d" % deploy.cfg.ROBOT_COMMAND_PORT,
    task             = EVAL_TASK_NAME,
)
print("save to:", EVAL_SAVE_ROOT / EVAL_TASK_NAME, "| 이미", len(list_episodes(EVAL_SAVE_ROOT / EVAL_TASK_NAME)), "개")

result = run_policy_with_keyboard_recording(deploy, recorder, duration_s=DURATION_S)
print("last state:", None if result is None else (result["state"], result["reason"]))


## 정리 / 비상 정지

`run()`은 끝나면(또는 예외/Ctrl-C 시) 알아서 `stop()`을 호출해 세션을 반납합니다. 이 셀은 카메라 소켓까지 완전히 닫는 용도입니다 — 다시 실행하려면 `PolicyDeployment`를 새로 만드세요.

셀이 멈춘 것처럼 보이거나 즉시 멈춰야 할 때는 커널 인터럽트(Ctrl-C / 정지 버튼)를 누르세요.

In [ ]:
deploy.close()
print("session closed")
